Proyecto_Analitica
Limpieza y organización de datos.

PASO 1: Importar librerias y cargar base de datos

In [3]:
import pandas as pd
import numpy as np

In [4]:
df_financiacion = pd.read_excel('/Users/yedisoncuervo/Downloads/BD taller clasificación (2) (2).xlsx')
df_financiacion.head(5)

,Caso,Perfil,Estado,Edad,Genero,ScoreCrediticio,PorcentajeFinanciacion,Plazo,IngresoEstimado,Gastos,CapacidadDePago,ValorCuotaMensual,M3_30AC
0,1004991730,ASALARIADO,NUEVO,30,FEMENINO,748,0.6850,72,3289800.0,2430508.51,0.361093,2379693,0
1,1005097331,INDEPENDIENTE,NUEVO,46,MASCULINO,670,0.2783,60,2425440.0,1621788.08,0.948770,847046,0
2,1005120587,INDEPENDIENTE,USADO,39,MASCULINO,752,1.0000,60,30000000.0,3614018.63,12.009213,2197145,0
3,1005152562,ASALARIADO,USADO,38,FEMENINO,758,1.0000,84,1631331.0,1725244.99,-0.068706,1366896,0
4,1005153782,INDEPENDIENTE,NUEVO,60,FEMENINO,846,0.6596,72,20907400.0,3439341.88,13.004595,1343222,0


PASO 2: Informacion general de la base de datos.

In [5]:
df_financiacion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21091 entries, 0 to 21090
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Caso                    21091 non-null  int64  
 1   Perfil                  21091 non-null  object 
 2   Estado                  21091 non-null  object 
 3   Edad                    21091 non-null  int64  
 4   Genero                  21091 non-null  object 
 5   ScoreCrediticio         21091 non-null  int64  
 6   PorcentajeFinanciacion  21091 non-null  float64
 7   Plazo                   21091 non-null  int64  
 8   IngresoEstimado         21063 non-null  float64
 9   Gastos                  21091 non-null  float64
 10  CapacidadDePago         21063 non-null  float64
 11  ValorCuotaMensual       21091 non-null  int64  
 12  M3_30AC                 21091 non-null  int64  
dtypes: float64(4), int64(6), object(3)
memory usage: 2.1+ MB


PASO 3: Validar si hay datos, faltantes, nulos Duplicados.
IMPORTANTE! Realizaremos si la cantidad de datos faltantes en una columna "Variable" supera eld 40% dicha columna será eliminada. 

In [6]:
# ── 4. Datos nulos ────────────────────────────────────────────────────────
print("DATOS NULOS")
nulos = df_financiacion.isnull().sum()
pct   = (nulos / len(df_financiacion) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct})
resumen_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0]
if resumen_nulos.empty:
    print("No hay datos nulos.")
else:
    print(resumen_nulos)

DATOS NULOS
                 Nulos  Porcentaje (%)
IngresoEstimado     28            0.13
CapacidadDePago     28            0.13


Dado que el porcentaje  de los datos nulos de las variables (IngresosEstimado/CapacidadDePago) Es de tan solo el 0.13% se procede a eliminar estas filas de nuestra BD.

In [7]:
# Eliminar filas con datos nulos
df_financiacion = df_financiacion.dropna()

In [8]:
# Verificar que quedaron cero nulos
print(f"Filas después de eliminar nulos: {len(df_financiacion)}")
print(f"Nulos restantes: {df_financiacion.isnull().sum().sum()}")

Filas después de eliminar nulos: 21063
Nulos restantes: 0


In [9]:
# ── 5. Datos duplicados ───────────────────────────────────────────────────
print("DATOS DUPLICADOS")
duplicados = df_financiacion.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")

DATOS DUPLICADOS
Filas duplicadas: 0


In [10]:
# ── 6. Estadísticas descriptivas ──────────────────────────────────────────
print("ESTADÍSTICAS DESCRIPTIVAS")
print(df_financiacion.describe())

ESTADÍSTICAS DESCRIPTIVAS
               Caso          Edad  ScoreCrediticio  PorcentajeFinanciacion  \
count  2.106300e+04  21063.000000     21063.000000            21063.000000   
mean   1.006178e+09     44.542563       782.361724                0.743488   
std    3.264530e+05     12.744980        85.314167                0.246626   
min    1.004992e+09     19.000000       343.000000                0.100000   
25%    1.005912e+09     34.000000       726.000000                0.552200   
50%    1.006159e+09     43.000000       783.000000                0.800000   
75%    1.006453e+09     54.000000       838.000000                1.000000   
max    1.006786e+09     75.000000       999.000000                1.067000   

              Plazo  IngresoEstimado        Gastos  CapacidadDePago  \
count  21063.000000     2.106300e+04  2.106300e+04     2.106300e+04   
mean      60.633101     5.018901e+06  1.142708e+08    -8.077237e+01   
std       12.497081     5.955286e+06  1.624658e+10     1.1

## IMPORTANTE! 
Con este codigo podremos saber si nuestra variable objetivo esta desbalanceada o no. 

In [11]:
# ── 7. Distribución de la variable objetivo ───────────────────────────────
print("DISTRIBUCIÓN VARIABLE OBJETIVO (M3_30AC)")
conteo = df_financiacion['M3_30AC'].value_counts()
pct_target = (conteo / len(df_financiacion) * 100).round(2)
print(pd.DataFrame({'Conteo': conteo, 'Porcentaje (%)': pct_target}))

DISTRIBUCIÓN VARIABLE OBJETIVO (M3_30AC)
         Conteo  Porcentaje (%)
M3_30AC                        
0         20228           96.04
1           835            3.96


## Balanceo de clases

Una vez depurada la base de datos — sin valores nulos, faltantes ni duplicados — se identificó 
un desbalance severo en la variable objetivo `M3_30AC`: el 96% de los registros corresponde 
a clientes sin mora (clase 0) y solo el 4% a clientes con mora (clase 1), lo que representa 
una proporción aproximada de 24:1.

Para mitigar este problema se aplica una estrategia de **undersampling aleatorio**: se conservan 
la totalidad de los 837 registros de la clase minoritaria (clase 1) y se extrae una muestra 
aleatoria de la clase mayoritaria en una proporción 1:2, es decir, 1.674 registros de la 
clase 0.

In [12]:
# ── Balanceo por undersampling (ratio 1:2) ────────────────────────────────
clase_1 = df_financiacion[df_financiacion['M3_30AC'] == 1]
clase_0 = df_financiacion[df_financiacion['M3_30AC'] == 0]

# Muestra aleatoria de clase 0 en proporción 1:2
clase_0_muestra = clase_0.sample(n=len(clase_1) * 2, random_state=42)

# Unir y mezclar
df_balanceado = pd.concat([clase_1, clase_0_muestra]).sample(frac=1, random_state=42).reset_index(drop=True)

# Verificar resultado
print(f"Total registros: {len(df_balanceado)}")
print(f"\nDistribución variable objetivo:")
conteo = df_balanceado['M3_30AC'].value_counts()
pct    = (conteo / len(df_balanceado) * 100).round(2)
print(pd.DataFrame({'Conteo': conteo, 'Porcentaje (%)': pct}))

Total registros: 2505

Distribución variable objetivo:
         Conteo  Porcentaje (%)
M3_30AC                        
0          1670           66.67
1           835           33.33
